# Notebook #2: Quality Control

Notebook performs QC on sc object from Notebook #1. Doublets are removed and cells with high mito, ribo, and hemo genes.

A final QC summary is downloaded at the end of the notebook and the cleaned object is saved for downstream processing.

In [35]:
!pip -q scanpy seaborn pandas numpy matplotlib session_info

ERROR: unknown command "scanpy"


In [36]:
# -- Load libraries
from pathlib import Path
import scanpy as sc
import seaborn as sns
import pandas as pd
import numpy as np
import session_info

from matplotlib import pyplot as plt

In [37]:
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [38]:
# -- Paths
project_dir = Path(
    "/content/drive/MyDrive/endo-immune-atlas"
)

dataset = "GSE179640"

interim_data_dir = (
    project_dir
    / "data"
    / "interim"
    / dataset
)

qc_results_dir = (
    project_dir
    / "results"
    / dataset
    / "qc"
)

qc_figures_dir = (
    project_dir
    / "figures"
    / dataset
    / "qc"
)



interim_data_dir.mkdir(
    parents=True,
    exist_ok=True,
)

qc_results_dir.mkdir(
    parents=True,
    exist_ok=True,
)

qc_figures_dir.mkdir(
    parents=True,
    exist_ok=True,
)

input_file = (
    interim_data_dir
    / "raw.h5ad"
)

In [39]:
# -- Parameters
min_genes = 500
min_UMI = 1000
max_UMI = 100000
mt_threshold = 25
min_cells = 3 # genes needed per cell to pass filter

In [40]:
# -- Set palettes
patient_colors_list = [
    sns.husl_palette(n_colors=1, h=h, s=0.9, l=0.65).as_hex()[0]
    for h in np.linspace(0.01, 0.70, 14)
]

# tissue types
tissue_palette = {
    "Ctrl": "#A3C5B7",
    "EuE": "#373F70",
    "EcP": "#596C94",
    "EcO": "#3B6341"
}

# -- Map to patient IDs
patient_ids = ["C01", "C02", "C03",
               "E01", "E02", "E03", "E04", "E05",
               "E06", "E07", "E08", "E09", "E10", "E11"]

patient_palette = dict(zip(patient_ids, patient_colors_list))

# Tissue order
tissue_order = [
    "Ctrl",
    "EuE",
    "EcP",
    "EcO",
]

In [41]:
# -- Load data
combined = sc.read_h5ad(input_file)

print(combined)
print(f"Starting cells: {combined.n_obs}")
print(f"Starting genes: {combined.n_vars}")

AnnData object with n_obs × n_vars = 119560 × 38224
    obs: 'sample_id', 'patient_id', 'tissue_type', 'condition', 'lesion_site', 'dataset'
    layers: None (.X)
Starting cells: 119560
Starting genes: 38224


In [42]:
# -- Preserve raw counts for downstream .. lesson learned
combined.layers["counts"] = combined.X.copy()

In [43]:
# -- QC gene groups

# hemoglobin genes (hemos)
combined.var["hemos"] = combined.var_names.str.contains("^HB[^(P)]")

# ribosomal genes (ribos)
combined.var["ribos"] = combined.var_names.str.startswith(("RPS", "RPL"))

# mitochondrial genes (mitos)
combined.var["mt"] = combined.var_names.str.contains("MT-")

In [44]:
# -- Calculate initial QC metrics
sc.pp.calculate_qc_metrics(combined,
                           qc_vars = ["mt", "ribos", "hemos"
                            ],
                           percent_top = None,
                           inplace = True,
                           log1p= False)


# Save summary
initial_qc_summary = (
    combined.obs
    .groupby(
        "sample_id",
        observed=True,
    )
    .agg(
        n_cells=("sample_id", "size"),
        median_genes=("n_genes_by_counts", "median"),
        median_counts=("total_counts", "median"),
        median_pct_mt=("pct_counts_mt", "median"),
    )
    .reset_index()
)

initial_qc_summary.to_csv(
    qc_results_dir / "02_qc_summary_before_filtering.csv",
    index=False,
)

display(initial_qc_summary)

,sample_id,n_cells,median_genes,median_counts,median_pct_mt
0,GSM6102532_C01_Ctrl,4133,4032.0,15243.0,15.595176
1,GSM6102533_C02_Ctrl,7801,2743.0,9214.0,12.777779
2,GSM6102534_C03_Ctrl,5531,2940.0,8997.0,9.656883
3,GSM6102537_E01_EuE,7665,2101.0,5612.0,13.657331
4,GSM6102540_E02_EuE,5277,4379.0,19144.0,11.266568
5,GSM6102543_E03_EuE,5302,2986.0,9657.5,12.457188
6,GSM6102546_E04_EuE,3819,2127.0,6975.0,11.697548
7,GSM6102549_E05_EuE,3030,1973.0,5226.5,9.615817
8,GSM6102550_E06_EcP,2494,1893.0,4978.0,6.322645
9,GSM6102551_E06_EuE,5226,3334.0,10890.5,11.644820


In [45]:
# -- Preprocessing figures
sc.pl.violin(combined,
    ["n_genes_by_counts"],
    groupby = "patient_id",
    jitter=0.4,
    rotation = 45,
    palette = patient_palette,
    ylabel = "Number of Genes by Counts",
    xlabel = "Patient_ID",
    show = False
)

plt.savefig(
    qc_figures_dir / "02_qc_n_genes_by_counts_by_patient.png",
    bbox_inches='tight',
    dpi=300
)

plt.close()


sc.pl.violin(combined,
    ["total_counts"],
    groupby = "patient_id",
    jitter=0.4,
    rotation = 45,
    palette = patient_palette,
    ylabel = "Total Counts",
    xlabel = "Patient_ID",
    show = False
)

plt.savefig(
    qc_figures_dir / "02_qc_total_counts_by_patient.png",
    bbox_inches='tight',
    dpi=300
)

plt.close()

sc.pl.violin(combined,
    ["pct_counts_mt"],
    groupby = "patient_id",
    jitter=0.4,
    rotation = 45,
    palette = patient_palette,
    ylabel = "% counts Mitochondrial",
    xlabel = "Patient_ID",
    show = False
)

plt.savefig(
    qc_figures_dir / "02_qc_pct_counts_mt_by_patient.png",
    bbox_inches='tight',
    dpi=300
)

plt.close()

/usr/local/lib/python3.12/dist-packages/scanpy/plotting/_anndata.py:997: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(
/usr/local/lib/python3.12/dist-packages/scanpy/plotting/_anndata.py:997: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(
/usr/local/lib/python3.12/dist-packages/scanpy/plotting/_anndata.py:997: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(


In [46]:
plt.scatter(
    combined.obs["n_genes_by_counts"],
    combined.obs["pct_counts_mt"],
    s=0.5,
    alpha=0.3
)
plt.xlabel("Number of Genes by Counts")
plt.ylabel("% counts Mitochondrial")
plt.axhline(y=25, color="red", linestyle="--")

plt.savefig(
    qc_figures_dir /"02_qc_mito_scatter.png",
    bbox_inches='tight',
    dpi=300
)
plt.close()

In [47]:
# -- Checking mt distribution to check on how to deal with QC parameters

samples = combined.obs["sample_id"].unique()
fig, axes = plt.subplots(4, 6, figsize=(20, 14))
axes = axes.flatten()

for i, sample in enumerate(sorted(samples)):
    mask = combined.obs["sample_id"] == sample
    axes[i].scatter(
        combined.obs.loc[mask, "n_genes_by_counts"],
        combined.obs.loc[mask, "pct_counts_mt"],
        s=0.5,
        alpha=0.3
    )
    axes[i].axhline(y=25, color="red", linestyle="--", linewidth=0.8)
    axes[i].set_title(sample, fontsize=8)
    axes[i].set_xlabel("Number of Genes", fontsize=7)
    axes[i].set_ylabel("% counts Mitochondrial", fontsize=7)

plt.tight_layout()

plt.savefig(
    qc_figures_dir / "02_qc_mito_scatter_per_sample_id.png",
    bbox_inches='tight',
    dpi=300
)

plt.close()

In [48]:
# -- Filter low quality cells
sc.pp.filter_cells(combined, min_genes=min_genes)
sc.pp.filter_cells(combined, min_counts=min_UMI)
sc.pp.filter_cells(combined, max_counts=max_UMI)

In [49]:
# -- Remove cells with high mt %
cells_before_filtering = combined.n_obs

combined.obs["outlier_mt"] = combined.obs["pct_counts_mt"] > mt_threshold

combined = combined[~combined.obs["outlier_mt"]].copy()


print(
    "Cells removed by count and mitochondrial filtering: "
    f"{cells_before_filtering - combined.n_obs}"
)

print(
    f"Cells remaining: {combined.n_obs}"
)

Cells removed by count and mitochondrial filtering: 11223
Cells remaining: 96031


In [50]:
# -- Doublet removal
cells_before_scrublet = combined.n_obs

sc.pp.scrublet(combined, batch_key="sample_id")


combined = combined[~combined.obs["predicted_doublet"]].copy()

print(
    "Predicted doublets removed: "
    f"{cells_before_scrublet - combined.n_obs}"
)

print(
    f"Cells remaining: {combined.n_obs}"
)

Predicted doublets removed: 1131
Cells remaining: 94900


In [51]:
# -- Filter genes
genes_before_filtering = combined.n_vars

sc.pp.filter_genes(combined, min_cells = min_cells)

print(
    "Genes removed: "
    f"{genes_before_filtering - combined.n_vars}"
)


print(f"Cells remaining: {combined.n_obs}")

Genes removed: 7317
Cells remaining: 94900


In [52]:
# -- Recalculate QC metrics after filtering
sc.pp.calculate_qc_metrics(
    combined,
    qc_vars=[
        "mt",
        "ribos",
        "hemos",
    ],
    percent_top=None,
    inplace=True,
    log1p=False,
)

In [53]:
# -- Save final QC summary
final_qc_summary = (
    combined.obs
    .groupby(
        "sample_id",
        observed=True,
    )
    .agg(
        n_cells=("sample_id", "size"),
        median_genes=("n_genes_by_counts", "median"),
        median_counts=("total_counts", "median"),
        median_pct_mt=("pct_counts_mt", "median"),
    )
    .reset_index()
)

final_qc_summary.to_csv(
    qc_results_dir / "02_qc_summary_after_filtering.csv",
    index=False,
)

display(final_qc_summary)

,sample_id,n_cells,median_genes,median_counts,median_pct_mt
0,GSM6102532_C01_Ctrl,2913,4801.0,21201.0,13.171259
1,GSM6102533_C02_Ctrl,6246,2977.0,10559.0,12.328665
2,GSM6102534_C03_Ctrl,4840,3061.0,9632.0,9.084982
3,GSM6102537_E01_EuE,4817,3206.0,9476.0,10.927835
4,GSM6102540_E02_EuE,3997,4813.0,23016.0,10.287180
5,GSM6102543_E03_EuE,3926,3454.0,12041.0,11.043671
6,GSM6102546_E04_EuE,3095,2490.0,8040.0,10.769557
7,GSM6102549_E05_EuE,2391,2507.0,7213.0,8.977084
8,GSM6102550_E06_EcP,1950,2279.5,6405.0,5.605155
9,GSM6102551_E06_EuE,4007,3747.0,13291.0,10.473383


In [54]:
# -- Save QC object for integration
output_file = (
    interim_data_dir
    / "qc.h5ad"
)

combined.write_h5ad(
    output_file
)

print("\nQC complete.")
print(f"Final cells: {combined.n_obs}")
print(f"Final genes: {combined.n_vars}")
print(f"Saved QC object to:\n{output_file}")


QC complete.
Final cells: 94900
Final genes: 30907
Saved QC object to:
/content/drive/MyDrive/endo-immune-atlas/data/interim/GSE179640/qc.h5ad


In [55]:
#-- Session info
session_info.show()

/usr/local/lib/python3.12/dist-packages/session_info/main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  mod_version = _find_version(mod.__version__)
/usr/local/lib/python3.12/dist-packages/session_info/main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  mod_version = _find_version(mod.__version__)
